In [15]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

In [16]:
df = pd.read_csv('../../Datos/df_general.csv')

In [17]:
# 2. Definir el Spoofing_ID que queremos aislar (ataque "Zero-Day")
isolated_spoof_id = 'A14'

print(f"--- INICIANDO EXPERIMENTO LEAVE-ONE-ATTACK-OUT ---")
print(f"Aislando el ataque: {isolated_spoof_id} para el Test Set\n")

# 3. Separar las muestras reales de las falsas
# Asumimos que la columna 'Key' contiene 'spoof' y algo como 'bonafide' o 'real'
df_reales = df[df['Key'] != 'spoof']
df_falsos = df[df['Key'] == 'spoof']

# 4. Preparar la bolsa de Test (Ataque aislado + 20% de audios reales)
reales_train_pool, reales_test = train_test_split(df_reales, test_size=0.2, random_state=42)
falsos_test = df_falsos[df_falsos['Spoofing_ID'] == isolated_spoof_id]

# 5. Preparar la bolsa de Train (El resto de ataques + 80% de audios reales)
falsos_train_pool = df_falsos[df_falsos['Spoofing_ID'] != isolated_spoof_id]

df_train_pool = pd.concat([reales_train_pool, falsos_train_pool])
df_test = pd.concat([reales_test, falsos_test])

# 6. Separar Features (X) y Target (y) ANTES del Undersampling
columnas_a_excluir = ['file_name', 'User_ID', 'Spoofing_ID', 'Key']

X_train_pool = df_train_pool.drop(columns=columnas_a_excluir)
y_train_pool = df_train_pool['Key']

X_test = df_test.drop(columns=columnas_a_excluir)
y_test = df_test['Key']

# 7. APLICAR UNDERSAMPLING ESTANDARIZADO
# Definimos exactamente cuántas muestras queremos de cada clase (1000 reales, 1000 falsos)
# Asumiendo que las clases en la columna 'Key' se llaman 'spoof' y la otra 'bonafide' (o similar)
clase_real = df_reales['Key'].iloc[0] 
estrategia_muestreo = {'spoof': 1000, clase_real: 1000}

--- INICIANDO EXPERIMENTO LEAVE-ONE-ATTACK-OUT ---
Aislando el ataque: A14 para el Test Set



In [ ]:
# Undersampler
rus = RandomUnderSampler(sampling_strategy=estrategia_muestreo, random_state=42)

X_train_resampled, y_train_resampled = rus.fit_resample(X_train_pool, y_train_pool)

print(f"\nDistribución del conjunto de Entrenamiento Tras Undersampling:")
print(y_train_resampled.value_counts())
print(f"Total muestras entrenamiento: {len(y_train_resampled)}")

print(f"\nDistribución del conjunto de Prueba (Test):")
print(y_test.value_counts())
print("-" * 50)


Distribución del conjunto de Entrenamiento Tras Undersampling:
Key
bonafide    1000
spoof       1000
Name: count, dtype: int64
Total muestras entrenamiento: 2000

Distribución del conjunto de Prueba (Test):
Key
spoof       4914
bonafide    1590
Name: count, dtype: int64
--------------------------------------------------


In [19]:
# 8. Instanciar y entrenar el Random Forest
rf_model = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)

print("\nEntrenando Random Forest con el dataset balanceado...")
rf_model.fit(X_train_resampled, y_train_resampled)

# 9. Realizar predicciones
print("Realizando predicciones...\n")
y_pred = rf_model.predict(X_test)

# 10. Evaluar resultados
print("=== RESULTADOS DEL MODELO ===")
print(f"Precisión global (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")

print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))


Entrenando Random Forest con el dataset balanceado...
Realizando predicciones...

=== RESULTADOS DEL MODELO ===
Precisión global (Accuracy): 0.9517

Reporte de Clasificación:
              precision    recall  f1-score   support

    bonafide       0.94      0.86      0.90      1590
       spoof       0.95      0.98      0.97      4914

    accuracy                           0.95      6504
   macro avg       0.95      0.92      0.93      6504
weighted avg       0.95      0.95      0.95      6504

Matriz de Confusión:
[[1362  228]
 [  86 4828]]
